# TRACE in PyHealth: nowcasting the next blood draw within a hospital visit

<a href="https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/trace_mimic4_lab_nowcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the laboratory measurement prediction setting of

> Yuyang Liang, Yankai Chen, Yixiang Fang, Laks V. S. Lakshmanan, Chenhao Ma.
> **TRACE: Intra-visit Clinical Event Nowcasting via Effective Patient Trajectory Encoding.** WWW 2025 Companion (short paper).
> [arXiv:2503.23072](https://arxiv.org/abs/2503.23072) · [doi:10.1145/3701716.3715545](https://doi.org/10.1145/3701716.3715545) · [authors' code](https://github.com/Amehi/TRACE)

with PyHealth:

- **Task** (`LabNowcastingMIMIC4`): an admission is a time-ordered sequence of events: laboratory results coded with their flag (`LAB:<itemid>:H/L/N`, from the reference range) and medications (`RX:<drug>`). From the events before the admission's last blood draw, predict the set of laboratory events (test and flag) of that draw. This is multi-label classification over the `(test, flag)` vocabulary.
- **Model** (`TRACE`): event embeddings plus a timestamp embedding with a decay and a daily periodic part, three time-aware Transformer layers with a learnable smoothed attention mask, and a mask token whose final state predicts the draw.
- **Metrics**: micro F1 at threshold 0.5, micro PR-AUC, Precision@5 and NDCG@5 (`pyhealth.metrics.multilabel_metrics_fn`).

The authors' repository has no license, so `TRACE` is a reimplementation from the paper; the repository was read only to confirm details (sizes, the 100-event window, time scales).

Part 1 runs end to end on the open **MIMIC-IV Clinical Database Demo** (100 patients, no credentials) on a CPU in a few minutes. It is a **smoke test**: with about 250 admissions, the test split holds about 40 samples, so its numbers say only that the pipeline works, not how TRACE compares with the paper. Set `USE_FULL_MIMIC4 = True` (Part 2) to run the same code on credentialed MIMIC-IV.

## Setup

The install line needs the TRACE pull request to be merged into `sunlabuiuc/PyHealth`; until then, install from the PR branch instead. It runs only on Colab; elsewhere, install PyHealth yourself first.

In [1]:
import sys

if "google.colab" in sys.modules:
    !pip install -q "git+https://github.com/sunlabuiuc/PyHealth"

In [2]:
import logging
import os
import random
import time
import urllib.request

os.environ["TQDM_DISABLE"] = "1"  # keep the Trainer's progress bars out of the output

import numpy as np
import torch

from pyhealth.datasets import MIMIC4EHRDataset, get_dataloader, split_by_patient
from pyhealth.metrics import multilabel_metrics_fn
from pyhealth.models import TRACE
from pyhealth.tasks import LabNowcastingMIMIC4
from pyhealth.trainer import Trainer

logging.getLogger("pyhealth").setLevel(logging.WARNING)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Data: the open demo, or full MIMIC-IV

**Switch:** set `USE_FULL_MIMIC4 = True` and point `MIMIC4_ROOT` at your MIMIC-IV copy (the folder with `hosp/`), e.g. on a mounted Google Drive, to run on the full data. MIMIC-IV needs PhysioNet credentialing ([physionet.org/content/mimiciv](https://physionet.org/content/mimiciv/)); use a GPU runtime. Otherwise the cell downloads the demo tables the task needs (about 3 MB).

In [3]:
USE_FULL_MIMIC4 = False                              # <- the switch
MIMIC4_ROOT = "/content/drive/MyDrive/mimic-iv/2.2"  # used when USE_FULL_MIMIC4

if USE_FULL_MIMIC4:
    root, cache_dir = MIMIC4_ROOT, "cache-trace-mimic4"
else:
    root, cache_dir = "mimic-iv-demo-2.2", "cache-trace-demo"
    base = "https://physionet.org/files/mimic-iv-demo/2.2/"
    for table in ["hosp/admissions", "hosp/patients", "hosp/labevents",
                  "hosp/d_labitems", "hosp/prescriptions", "icu/icustays"]:
        path = os.path.join(root, table + ".csv.gz")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(base + table + ".csv.gz", path)
print("data:", root)

data: mimic-iv-demo-2.2


## Build the task

`LabNowcastingMIMIC4()` uses the paper's setup: one sample per admission (its last blood draw), the 99 most recent earlier events (the model appends the mask token to make 100), `labevents` and `prescriptions`. `event_times` holds the hours from each event to the draw. `target="all"` would give one sample per draw instead.

In [4]:
t0 = time.time()
dataset = MIMIC4EHRDataset(root=root, tables=["labevents", "prescriptions"],
                           cache_dir=cache_dir)
samples = dataset.set_task(LabNowcastingMIMIC4())

events_vocab = samples.input_processors["events"].size()
labels = samples.output_processors["labs"]
n_true = np.array([float(samples[i]["labs"].sum()) for i in range(len(samples))])
print(f"{len(samples)} samples in {time.time() - t0:.0f} s; "
      f"{events_vocab} input codes, {labels.size()} target (test, flag) labels")
print(f"labels per sample: {n_true.mean():.2f} ± {n_true.std():.2f}"
      "  (paper's MIMIC-IV: 18.21 ± 9.97)")
example = samples[0]
vocab = {i: c for c, i in labels.label_vocab.items()}
print("example target:", [vocab[i] for i in np.flatnonzero(example["labs"])][:8], "...")

249 samples in 0 s; 872 input codes, 302 target (test, flag) labels
labels per sample: 20.44 ± 11.69  (paper's MIMIC-IV: 18.21 ± 9.97)
example target: ['LAB:50835:N', 'LAB:50843:N', 'LAB:50849:N', 'LAB:51114:H', 'LAB:51116:H', 'LAB:51117:H', 'LAB:51118:H', 'LAB:51120:N'] ...


## Train and evaluate

The paper splits samples 75/10/15 at random; here the split is by patient, so no patient is in two splits. TRACE uses the paper's settings: `d = 256`, 3 layers, 4 heads, dropout 0.1, Adam with learning rate 1e-3, batch size 128, 25 epochs, keeping the epoch with the best validation PR-AUC, and denoising weight 0.01.

The F1 threshold is 0.5, as in the paper. The authors' released code (`recalls_and_ndcgs_for_ks` in `model/trainer.py`) weights every rank by 1 / log2(2) = 1 when computing NDCG@k, which makes it `hits@k / min(#true, k)` (equal to Precision@5 whenever a draw has at least 5 tests); it is shown as `ndcg@5 (paper)` next to the standard NDCG@5 that `multilabel_ranking_at_k` computes. "Predicted per sample" is the number of labels above 0.5, which the paper compares with the true count (Table 2). A **label-frequency** baseline, which scores every sample with each label's training frequency, shows how much of the score comes from common tests alone.

In [5]:
SEEDS = [0, 1, 2]
EPOCHS = 25
METRICS = ["f1_micro", "pr_auc_micro", "precision@5", "ndcg@5"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def scores(y_true, y_prob):
    out = multilabel_metrics_fn(y_true, y_prob, metrics=METRICS, threshold=0.5)
    top5 = np.argsort(-y_prob, axis=1, kind="stable")[:, :5]
    hits = np.take_along_axis(y_true, top5, axis=1).sum(1)
    out["ndcg@5 (paper)"] = float(np.mean(hits / np.minimum(y_true.sum(1), 5)))
    out["predicted per sample"] = float((y_prob >= 0.5).sum(1).mean())
    return out


def labels_of(split):
    return np.stack([split[i]["labs"].numpy() for i in range(len(split))])


def label_frequency(split):
    total = sum(split[i]["labs"].numpy() for i in range(len(split)))
    return total / len(split)


results = {"TRACE": [], "label frequency": []}
for seed in SEEDS:
    seed_everything(seed)
    train, val, test = split_by_patient(samples, [0.75, 0.10, 0.15], seed=seed)
    y_test = labels_of(test)
    frequency = label_frequency(train)
    results["label frequency"].append(scores(y_test, np.tile(frequency, (len(test), 1))))

    model = TRACE(samples)  # paper defaults
    trainer = Trainer(model, metrics=["pr_auc_micro"], device=device,
                      enable_logging=False)
    trainer.train(get_dataloader(train, batch_size=128, shuffle=True),
                  get_dataloader(val, batch_size=128),
                  epochs=EPOCHS, monitor="pr_auc_micro",
                  optimizer_params={"lr": 1e-3})
    y_true, y_prob, _ = trainer.inference(get_dataloader(test, batch_size=128))
    results["TRACE"].append(scores(y_true, y_prob))
    print(f"seed {seed}: {len(train)}/{len(val)}/{len(test)} samples, "
          f"TRACE test PR-AUC {results['TRACE'][-1]['pr_auc_micro']:.3f}")

seed 0: 178/36/35 samples, TRACE test PR-AUC 0.535


seed 1: 201/23/25 samples, TRACE test PR-AUC 0.596


seed 2: 201/20/28 samples, TRACE test PR-AUC 0.532


## Paper vs. this run

TRACE's MIMIC-IV results in the paper (Table 1, mean ± sd over runs; full MIMIC-IV, random 75/10/15 split), next to the numbers obtained above (mean ± sd over the seeds).

In [6]:
PAPER_MIMIC4 = {"f1_micro": "0.8535 ± 0.0062", "pr_auc_micro": "0.7777 ± 0.0021",
                "precision@5": "0.7566 ± 0.0023", "ndcg@5 (paper)": "0.7960 ± 0.0031",
                "predicted per sample": "20.49 ± 1.06"}
data_name = "full MIMIC-IV" if USE_FULL_MIMIC4 else "MIMIC-IV demo (smoke test)"


def mean_sd(rows, key):
    values = np.array([r[key] for r in rows])
    digits = 2 if key == "predicted per sample" else 4
    return f"{values.mean():.{digits}f} ± {values.std():.{digits}f}"


keys = ["f1_micro", "pr_auc_micro", "precision@5", "ndcg@5 (paper)", "ndcg@5",
        "predicted per sample"]
print(f"{'metric':22s} {'paper (MIMIC-IV)':>18s} {'TRACE, ' + data_name:>36s}"
      f" {'label frequency':>18s}")
for key in keys:
    print(f"{key:22s} {PAPER_MIMIC4.get(key, '-'):>18s} "
          f"{mean_sd(results['TRACE'], key):>36s} "
          f"{mean_sd(results['label frequency'], key):>18s}")

metric                   paper (MIMIC-IV)    TRACE, MIMIC-IV demo (smoke test)    label frequency
f1_micro                  0.8535 ± 0.0062                      0.5052 ± 0.0112    0.4719 ± 0.0240
pr_auc_micro              0.7777 ± 0.0021                      0.5544 ± 0.0294    0.5006 ± 0.0204
precision@5               0.7566 ± 0.0023                      0.6510 ± 0.0233    0.6744 ± 0.0204
ndcg@5 (paper)            0.7960 ± 0.0031                      0.6510 ± 0.0233    0.6744 ± 0.0204
ndcg@5                                  -                      0.6573 ± 0.0167    0.6857 ± 0.0186
predicted per sample         20.49 ± 1.06                         11.87 ± 1.35       11.67 ± 0.47


## Reading the numbers

- **On the demo this is a smoke test.** About 200 training admissions are far too few for a 3-layer, 256-wide Transformer over about 300 `(test, flag)` labels and 900 input codes, and the test split has about 30 samples. On the demo TRACE lands close to the label-frequency baseline (a little higher on F1 and PR-AUC, a little lower on Precision@5), and well below the paper. Neither says anything about how TRACE compares with the paper; that needs the full data.
- **To compare with the paper**, set `USE_FULL_MIMIC4 = True`. Building the event table reads `labevents` (about 120 million rows) once and caches it in `cache_dir`.
- **Known differences from the paper's setup**, which the authors may want to check:
  - The paper does not list its event tables or exclusions. Here they are `labevents` and `prescriptions`; ICU `procedureevents` can be added with `LabNowcastingMIMIC4(event_tables=("labevents", "prescriptions", "procedureevents"))` (the table must also be loaded). `procedures_icd` carries only a discharge time in PyHealth's config, so it cannot be ordered within a visit.
  - Lab flags: high/low/normal from `valuenum` and the reference range, falling back to the `flag` column (`A` for abnormal) when there is no range.
  - Event times are hours before the draw being predicted; the mask token has time 0.
  - The split is by patient; the paper's is random over samples.
  - The loss is the paper's binary cross-entropy (the released code uses a softmax cross-entropy over the flattened batch), and the denoising term is the paper's Frobenius norm of the smoothed masks.
  - Metrics are computed over the whole test set; the released code averages them over batches of 128.